# NS-3.48 environment setup

Run this notebook in a fresh Google Colab runtime. It mounts Drive, bootstraps only the missing setup source files from the hash-pinned `colab/stage2_setup_sources.zip` bundle if needed, checks dependencies, then restores a compatible build or builds NS-3.48 under `/content`. It does **not** run a research simulation, start the 180-run matrix, or upload research datasets.

The default Drive project path is `MyDrive/DISSERTATION`, matching the folder link provided for this setup. Set `STAGE2_PROJECT_DIR` to an exact path under `MyDrive` to use another folder. The notebook never creates a folder or silently falls back.

In [ ]:
from pathlib import Path
import hashlib
import io
import os
import sys
import tempfile
import zipfile

from google.colab import drive, files

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive').resolve()
PROJECT_DIR = Path(
    os.environ.get('STAGE2_PROJECT_DIR', str(DRIVE_ROOT / 'DISSERTATION'))
).expanduser().resolve()
if not PROJECT_DIR.is_relative_to(DRIVE_ROOT):
    raise ValueError(f'Project folder must be under {DRIVE_ROOT}: {PROJECT_DIR}')
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f'Drive project folder is missing: {PROJECT_DIR}. '
        'Set STAGE2_PROJECT_DIR to the exact existing project folder under MyDrive.'
    )
EXPECTED_BUNDLE_SHA256 = 'ae2165bf92537e3a1da25af572ee480104003da09e3e2c3656bb9d605a917da5'
BUNDLE_FILES = {
    'colab/__init__.py',
    'colab/environment.py',
    'requirements.txt',
    'ns3/dsr_instrumentation/README.md',
    'ns3/dsr_instrumentation/apply.sh',
    'ns3/dsr_instrumentation/manifest.json',
    'ns3/dsr_instrumentation/ns3-3.48-dsr.patch',
    'ns3/dsr_instrumentation/src/dsr/model/stage2-dsr-event-sink.h',
    'ns3/dsr_instrumentation/scratch/dsr_manet_dataset.cc',
}
missing_files = [relative for relative in sorted(BUNDLE_FILES) if not (PROJECT_DIR / relative).is_file()]
if missing_files:
    print('The linked Drive project is missing the setup source package.')
    print('Choose colab/stage2_setup_sources.zip from the local project when the upload button appears.')
    uploaded = files.upload()
    if set(uploaded) != {'stage2_setup_sources.zip'}:
        raise ValueError('Upload exactly one file named stage2_setup_sources.zip; no project data is needed.')
    bundle = uploaded['stage2_setup_sources.zip']
    if hashlib.sha256(bundle).hexdigest() != EXPECTED_BUNDLE_SHA256:
        raise ValueError('Setup source bundle SHA-256 mismatch; use the bundle from this project revision.')
    with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
        infos = [entry for entry in archive.infolist() if not entry.is_dir()]
        names = {entry.filename for entry in infos}
        if names != BUNDLE_FILES or len(infos) != len(BUNDLE_FILES):
            raise ValueError('Setup bundle contents do not match the approved source-file list.')
        pending_writes = []
        for entry in infos:
            relative = Path(entry.filename)
            if relative.is_absolute() or '..' in relative.parts:
                raise ValueError(f'Unsafe setup bundle path: {entry.filename}')
            destination = PROJECT_DIR / relative
            data = archive.read(entry)
            if destination.exists():
                if not destination.is_file() or hashlib.sha256(destination.read_bytes()).digest() != hashlib.sha256(data).digest():
                    raise FileExistsError(f'Refusing to overwrite a conflicting Drive file: {destination}')
            else:
                pending_writes.append((destination, data))
        for destination, data in pending_writes:
            destination.parent.mkdir(parents=True, exist_ok=True)
            with tempfile.NamedTemporaryFile(dir=destination.parent, delete=False) as staged:
                staged.write(data)
                staged_path = Path(staged.name)
            if destination.exists():
                staged_path.unlink()
                raise FileExistsError(f'Drive file appeared during setup; refusing to overwrite: {destination}')
            os.rename(staged_path, destination)
if any(not (PROJECT_DIR / relative).is_file() for relative in BUNDLE_FILES):
    raise FileNotFoundError('Setup source bundle did not provide every required file.')
sys.path.insert(0, str(PROJECT_DIR))
from colab.environment import verify_instrumentation_package
verify_instrumentation_package(PROJECT_DIR / 'ns3/dsr_instrumentation')
print(f'Project folder: {PROJECT_DIR}')

## Choose setup mode

`AUTO` reuses a compatible local/Drive build cache or builds if needed. `RESTORE` requires a matching cache. `BUILD` builds the pinned source. `VERIFY` is read-only and requires an already compatible build. Cache writes are opt-in and use one compressed archive, not many small Drive writes.

In [ ]:
if not (PROJECT_DIR / 'colab/environment.py').is_file():
    raise RuntimeError('Setup sources are missing. Run the first setup cell to upload and verify stage2_setup_sources.zip, then rerun this cell.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from colab.environment import setup_environment

MODE = os.environ.get('STAGE2_ENV_MODE', 'AUTO').upper()
NS3_DIR = Path('/content/ns-3.48')
CACHE_CANDIDATE = PROJECT_DIR / 'environment/build-cache'
BUILD_CACHE_DIR = CACHE_CANDIDATE if CACHE_CANDIDATE.is_dir() else None
SAVE_CACHE = os.environ.get('STAGE2_SAVE_BUILD_CACHE') == '1'
if SAVE_CACHE and BUILD_CACHE_DIR is None:
    raise FileNotFoundError(
        f'Create and verify the approved build-cache folder before saving: {CACHE_CANDIDATE}'
    )
JOBS = int(os.environ['STAGE2_BUILD_JOBS']) if 'STAGE2_BUILD_JOBS' in os.environ else None

REPORT = setup_environment(
    mode=MODE,
    project_dir=PROJECT_DIR,
    ns3_dir=NS3_DIR,
    build_cache_dir=BUILD_CACHE_DIR,
    build_jobs=JOBS,
    save_cache=SAVE_CACHE,
)
print('Environment status:', REPORT['status'])
print('NS-3 build status:', REPORT['ns3']['build_status'])
print('Build fingerprint:', REPORT['ns3']['build_fingerprint'])
print('Manifest:', PROJECT_DIR / 'COLAB_ENVIRONMENT_MANIFEST.json')